In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("priyankraval/nurse-stress-prediction-wearable-sensors")

print("Path to dataset files:", path)

100%|██████████| 76.2M/76.2M [00:00<00:00, 139MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/priyankraval/nurse-stress-prediction-wearable-sensors/versions/1


# Logistic Regression Pipeline
This notebook downloads the dataset, loads the main CSV, preprocesses features, trains a logistic regression model, evaluates it, and saves the trained model and scaler.

In [3]:
import os
import glob
import zipfile
import pandas as pd

# Determine where files are and find CSVs
if os.path.isdir(path):
    data_dir = path
elif os.path.isfile(path) and path.lower().endswith('.zip'):
    extract_dir = 'dataset_extracted'
    with zipfile.ZipFile(path, 'r') as zf:
        zf.extractall(extract_dir)
    data_dir = extract_dir
else:
    data_dir = os.path.dirname(path) or '.'

csv_files = glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True)
print('Found CSV files:', csv_files)

if not csv_files:
    raise FileNotFoundError('No CSV files found in downloaded dataset path')

# Load the first CSV as the main table (inspect others if needed)
df = pd.read_csv(csv_files[0])
print('Dataframe shape:', df.shape)
df.head()

Found CSV files: ['/root/.cache/kagglehub/datasets/priyankraval/nurse-stress-prediction-wearable-sensors/versions/1/merged_data.csv']


/tmp/ipykernel_18827/1808129244.py:24: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_files[0])


Dataframe shape: (11509051, 9)


,X,Y,Z,EDA,HR,TEMP,id,datetime,label
0,-13.0,-61.0,5.0,6.769995,99.43,31.17,15,2020-07-08 14:03:00.000000000,2.0
1,-20.0,-69.0,-3.0,6.769995,99.43,31.17,15,2020-07-08 14:03:00.031249920,2.0
2,-31.0,-78.0,-15.0,6.769995,99.43,31.17,15,2020-07-08 14:03:00.062500096,2.0
3,-47.0,-65.0,-38.0,6.769995,99.43,31.17,15,2020-07-08 14:03:00.093750016,2.0
4,-67.0,-57.0,-53.0,6.769995,99.43,31.17,15,2020-07-08 14:03:00.124999936,2.0


In [ ]:
# Basic exploration and preprocessing 
print('Columns:', list(df.columns))
print('\nMissing values per column:\n', df.isna().sum())
print('\nDtypes:\n', df.dtypes)

# Use provided label column 
if 'label' in df.columns:
    target_col = 'label'
else:
    possible_targets = [c for c in df.columns if c.lower() in ['stress','stress_level','target','class','activity','label_name','y']]
    target_col = possible_targets[0] if possible_targets else df.columns[-1]
    print('Using target column:', target_col)

# Select features
available = {c.lower(): c for c in df.columns}
feature_names = []
for name in ['eda','hr','temp']:
    if name in available:
        feature_names.append(available[name])
    else:
        raise KeyError(f'Required feature "{name}" not found in dataframe columns')

print('Using features:', feature_names)

# Keep id if present for group-aware splitting
group_ids = df['id'] if 'id' in df.columns else None
if group_ids is not None:
    print('Found', group_ids.nunique(), 'unique ids')

# Build X and y, drop rows with missing feature/label values
X = df[feature_names].copy()
y = df[target_col].copy()
data = pd.concat([X, y, df[['id']] if 'id' in df.columns else pd.DataFrame()], axis=1)
data = data.dropna(subset=feature_names + [target_col])
X = data[feature_names]
y = data[target_col]
if 'id' in data.columns:
    group_ids = data['id']
else:
    group_ids = None

# Ensure y is integer labels 0,1,2
y = pd.to_numeric(y, errors='coerce').astype(int)
print('Class distribution:\n', y.value_counts())
print('Final feature shape:', X.shape)


Columns: ['X', 'Y', 'Z', 'EDA', 'HR', 'TEMP', 'id', 'datetime', 'label']

Missing values per column:
 X           0
Y           0
Z           0
EDA         0
HR          0
TEMP        0
id          0
datetime    0
label       0
dtype: int64

Dtypes:
 X           float64
Y           float64
Z           float64
EDA         float64
HR          float64
TEMP        float64
id           object
datetime     object
label       float64
dtype: object
Using features: ['EDA', 'HR', 'TEMP']
Found 18 unique ids
Class distribution:
 label
2    8540583
0    2162246
1     806222
Name: count, dtype: int64
Final feature shape: (11509051, 3)


In [10]:
# Train-test split, scaling, model training, evaluation, and saving
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib
import numpy as np

# If group ids are present, perform a group-aware split so entire ids are held out
use_group_split = True if group_ids is not None else False
if use_group_split:
    # Prepare groups as a 1-D string array aligned with X
    if len(group_ids) != len(X):
        groups_arr = group_ids.loc[X.index].values.astype(str)
    else:
        groups_arr = group_ids.values.astype(str)

    # Try multiple random group splits until the test set contains all classes
    all_classes = set(y.unique())
    gss = GroupShuffleSplit(n_splits=1000, test_size=0.2, random_state=42)
    found = False
    attempts = 0
    for train_idx, test_idx in gss.split(X, y, groups=groups_arr):
        attempts += 1
        test_labels = set(y.iloc[test_idx].unique())
        if all_classes.issubset(test_labels):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            found = True
            print(f'Found group split with all classes in test after {attempts} attempts')
            break
    if not found:
        # Fallback: warn and use stratified split to guarantee class coverage
        print('Warning: could not find a group split that contains all classes in test. Falling back to stratified split (no group guarantee).')
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if len(all_classes) > 1 else None)
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if len(set(y))>1 else None)
    print('Performed random stratified split')

# Scale numeric features
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# Choose solver based on number of classes
n_classes = len(set(y))
if n_classes > 2:
    clf = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=200)
else:
    clf = LogisticRegression(solver='lbfgs', max_iter=200)

# Fit
clf.fit(X_train_s, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test_s)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification report:\n', classification_report(y_test, y_pred))
print('\nConfusion matrix:\n', confusion_matrix(y_test, y_pred))

# Save artifacts
artifacts = {'model': clf, 'scaler': scaler}
joblib.dump(artifacts, 'logreg_artifacts.joblib')
print('Saved model and scaler to logreg_artifacts.joblib')


Found group split with all classes in test after 2 attempts


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.8739250174927014


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Classification report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00    294631
           1       0.00      0.00      0.00     34562
           2       0.87      1.00      0.93   2281896

    accuracy                           0.87   2611089
   macro avg       0.29      0.33      0.31   2611089
weighted avg       0.76      0.87      0.82   2611089


Confusion matrix:
 [[      0       0  294631]
 [      0       0   34562]
 [      0       0 2281896]]
Saved model and scaler to logreg_artifacts.joblib


In [12]:
# Handle class imbalance: balanced weights, upsampling, and SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample
import joblib
import imblearn

print('Label counts (training set if available, else full):')
try:
    print(y.value_counts())
except Exception:
    print('Could not print y counts')

# Ensure we have train/test splits and scaled features
if 'X_train_s' not in globals() or 'X_test_s' not in globals():
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if len(set(y))>1 else None)
    scaler_temp = StandardScaler().fit(X_train)
    X_train_s = scaler_temp.transform(X_train)
    X_test_s = scaler_temp.transform(X_test)
    scaler = scaler_temp
else:
    # assume X_train, X_test, y_train, y_test, scaler exist
    pass

n_classes = len(set(y))

# 1) Class-weight balanced logistic regression
print('\nTraining LogisticRegression with class_weight=\'balanced\'...')
clf_bal = LogisticRegression(class_weight='balanced', multi_class='multinomial' if n_classes>2 else 'ovr', solver='lbfgs', max_iter=200)
clf_bal.fit(X_train_s, y_train)

y_pred_bal = clf_bal.predict(X_test_s)
print('Balanced model classes:', clf_bal.classes_)
print('Accuracy (balanced):', accuracy_score(y_test, y_pred_bal))
print('\nClassification report (balanced):\n', classification_report(y_test, y_pred_bal))
print('\nConfusion matrix (balanced):\n', confusion_matrix(y_test, y_pred_bal))
joblib.dump({'model': clf_bal, 'scaler': scaler}, 'logreg_balanced.joblib')
print('Saved balanced model to logreg_balanced.joblib')

# 2) Upsample minority classes in the training set (simple bootstrap upsampling)
print('\nUpsampling minority classes and retraining...')
train_df = X_train.copy()
train_df['__y__'] = y_train.values

counts = train_df['__y__'].value_counts()
print('Training class counts before upsampling:\n', counts)
max_count = counts.max()
resampled_frames = []
for cls, cnt in counts.items():
    cls_df = train_df[train_df['__y__'] == cls]
    if cnt < max_count:
        cls_upsampled = resample(cls_df, replace=True, n_samples=max_count, random_state=42)
        resampled_frames.append(cls_upsampled)
    else:
        resampled_frames.append(cls_df)

train_resampled = pd.concat(resampled_frames)
X_res = train_resampled.drop(columns=['__y__'])
Y_res = train_resampled['__y__']

# scale resampled features using existing scaler
X_res_s = scaler.transform(X_res)

clf_up = LogisticRegression(multi_class='multinomial' if n_classes>2 else 'ovr', solver='lbfgs', max_iter=200)
clf_up.fit(X_res_s, Y_res)

y_pred_up = clf_up.predict(X_test_s)
print('Accuracy (upsampled):', accuracy_score(y_test, y_pred_up))
print('\nClassification report (upsampled):\n', classification_report(y_test, y_pred_up))
print('\nConfusion matrix (upsampled):\n', confusion_matrix(y_test, y_pred_up))
joblib.dump({'model': clf_up, 'scaler': scaler}, 'logreg_upsampled.joblib')
print('Saved upsampled model to logreg_upsampled.joblib')

# 3) Try SMOTE if imblearn is available
print('\nAttempting SMOTE (if imblearn is installed)...')
try:
    from imblearn.over_sampling import SMOTE
    sm = SMOTE(random_state=42)
    X_train_scaled = scaler.transform(X_train)
    X_sm, y_sm = sm.fit_resample(X_train_scaled, y_train)
    clf_sm = LogisticRegression(multi_class='multinomial' if n_classes>2 else 'ovr', solver='lbfgs', max_iter=200)
    clf_sm.fit(X_sm, y_sm)
    y_pred_sm = clf_sm.predict(X_test_s)
    print('Accuracy (SMOTE):', accuracy_score(y_test, y_pred_sm))
    print('\nClassification report (SMOTE):\n', classification_report(y_test, y_pred_sm))
    print('\nConfusion matrix (SMOTE):\n', confusion_matrix(y_test, y_pred_sm))
    joblib.dump({'model': clf_sm, 'scaler': scaler}, 'logreg_smote.joblib')
    print('Saved SMOTE model to logreg_smote.joblib')
except Exception as e:
    print('SMOTE not available or failed:', e)
    print('You can install imbalanced-learn with: pip install imbalanced-learn')


Label counts (training set if available, else full):
label
2    8540583
0    2162246
1     806222
Name: count, dtype: int64

Training LogisticRegression with class_weight='balanced'...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Balanced model classes: [0 1 2]
Accuracy (balanced): 0.45015432258341254

Classification report (balanced):
               precision    recall  f1-score   support

           0       0.09      0.31      0.14    294631
           1       0.00      0.00      0.00     34562
           2       0.91      0.47      0.62   2281896

    accuracy                           0.45   2611089
   macro avg       0.33      0.26      0.26   2611089
weighted avg       0.80      0.45      0.56   2611089


Confusion matrix (balanced):
 [[  92769  107160   94702]
 [  19201       0   15361]
 [ 877049  322223 1082624]]
Saved balanced model to logreg_balanced.joblib

Upsampling minority classes and retraining...
Training class counts before upsampling:
 __y__
2    6258687
0    1867615
1     771660
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy (upsampled): 0.4494772104665908

Classification report (upsampled):
               precision    recall  f1-score   support

           0       0.09      0.31      0.14    294631
           1       0.00      0.00      0.00     34562
           2       0.91      0.47      0.62   2281896

    accuracy                           0.45   2611089
   macro avg       0.33      0.26      0.26   2611089
weighted avg       0.80      0.45      0.56   2611089


Confusion matrix (upsampled):
 [[  92769  107352   94510]
 [  19201       0   15361]
 [ 877089  323951 1080856]]
Saved upsampled model to logreg_upsampled.joblib

Attempting SMOTE (if imblearn is installed)...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy (SMOTE): 0.44973457434809766

Classification report (SMOTE):
               precision    recall  f1-score   support

           0       0.09      0.31      0.14    294631
           1       0.00      0.00      0.00     34562
           2       0.91      0.47      0.62   2281896

    accuracy                           0.45   2611089
   macro avg       0.33      0.26      0.26   2611089
weighted avg       0.80      0.45      0.56   2611089


Confusion matrix (SMOTE):
 [[  92769  107280   94582]
 [  19201       0   15361]
 [ 877073  323295 1081528]]
Saved SMOTE model to logreg_smote.joblib


The classifier has seen class‑1 during training (support >> 0) but the learned decision boundary never makes class‑1 the highest‑probability class for any test sample — so argmax(probabilities) never equals 1.

Likely causes:
The three features (EDA, HR, TEMP) are not separable enough for class‑1 — class‑1 points overlap with classes 0/2 in feature space.
Class imbalance and regularization push the model toward predicting the two dominant regions (0 or 2).
Model choice if class‑1 is nonlinearly separable.

Practical fixes to try (pick one or more)
Try a non-linear model (RandomForest or XGBoost) — often recovers minority class if features hold signal.